Business Problem :- An e-commerce company wants to test a redesigned checkout experience to determine whether it improves checkout conversion while maintaining revenue and payment quality.



## Step 1:

Our assumptions

We'll use:
- Baseline checkout conversion: 60%
- MDE: 5% relative
- Treatment conversion we want to detect: 63%
- Significance level (α): 0.05
- Power: 80%
- Two-sided test

So: Pa= 0.60  # current conversion rate

Pb=0.60 * 1.05 = 0.63 #1.05 is the 5% (1+0.05) increased conversion rate
 
The important distinction is:

5% MDE relative = 3 percentage-point absolute difference.

In [1]:
from scipy.stats import norm
norm.ppf(0.0)

-inf

For a two-proportion test, the approximate sample-size formula per group is: 

n=((Zα/2 * (2pˉ​(1−pˉ​))**0.5 + Zβ (pA(1-pA) + pB(1-pB))**0.5))**2 /  (pB​−pA​)*2 

Where:
- n = required sample size per group
- p_A = baseline/control conversion rate = 0.60
- p_B = expected treatment conversion rate= 0.63
- pˉ = ( p_A+p_B )/ 2 = pooled/average proportion 
- Z_alpha/2 = Z-value for significance level = 1.96
- Z_beta = Z-value corresponding to desired power = 0.84

In [6]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

baseline = 0.60
treatment = 0.63

effect_size = proportion_effectsize(
    baseline,
    treatment
)

sample_size = NormalIndPower().solve_power(
    effect_size=effect_size,
    alpha=0.05,
    power=0.80,
    ratio=1,
    alternative="two-sided"
)
print(sample_size)

4128.280959778545


## Step 2: Determine experiment duration

We'll calculate the daily number of eligible checkout users, then:

Experiment Duration = Required Total Users \ Average Daily Eligible Users 

- Control = 4,129
- Treatment = 4,129
- Total = 8,242 users
- Average eligible checkout users per day = 2400

Therefore, expected total checkout users: 1,200 * 14 = 16,800

With 50/50 randomization: 16,800/2=8,400

So we'd need approximately 14 days.
But there is an important A/B testing consideration

We shouldn't simply stop the moment we hit the required sample size.

We also want the experiment to cover normal weekly behavior.

For example, running only Monday–Thursday could give us a distorted picture because user behavior may differ on weekends.

So we'll generally want to run for at least one or more complete weekly cycles, while ensuring we reach the required sample size.

The e-commerce platform receives approximately 2,400 eligible checkout users per day. The experiment is planned for 14 days to capture two complete weekly cycles and collect sufficient sample size.

Step 3: Data Model & Experiment Randomization

Our experiment architecture
a) We'll use 5 core tables:

                    ┌──────────────────┐
                    │      users       │
                    │──────────────────│
                    │ user_id          │
                    │ signup_date      │
                    │ country          │
                    │ device           │
                    │ user_type        │
                    └────────┬─────────┘
                             │
                             │ user_id
                             ▼
              ┌──────────────────────────┐
              │ experiment_assignments   │
              │──────────────────────────│
              │ user_id                  │
              │ experiment_id            │
              │ experiment_group         │
              │ assigned_at              │
              └────────────┬─────────────┘
                           │
                           │ user_id
                           ▼
                    ┌──────────────┐
                    │    events    │
                    │──────────────│
                    │ event_id     │
                    │ user_id      │
                    │ event_name   │
                    │ timestamp    │
                    │ device       │
                    └──────┬───────┘
                           │
                           │ user_id
                           ▼
                    ┌──────────────┐
                    │    orders    │
                    │──────────────│
                    │ order_id     │
                    │ user_id      │
                    │ order_time   │
                    │ amount       │
                    │ payment_status│
                    │ refund_flag  │
                    └──────────────┘

                    ┌──────────────────┐
                    │   experiments    │
                    │──────────────────│
                    │ experiment_id    │
                    │ name             │
                    │ start_date       │
                    │ end_date         │
                    │ hypothesis       │
                    │ primary_metric   │
                    │ status           │
                    └──────────────────┘

This allows us to calculate:

Revenue per User = Total Revenue \ Exposed Users

AOV = Total Revenue \ Completed Orders

Payment Failure Rate= Failed Payments \ Total Payment Attempts

Refund Rate = Completed Orders / Refunded Orders
	

## Step 4: will be Randomization

In [3]:
#We'll use reproducible randomization with a seed,
#np.random.seed(42)
#Then randomly assign A/B.

#The seed is important because if we rerun the script, we get the same assignment.

#After randomization, we'll perform an important validation:

Sample Ratio Mismatch (SRM)

We'll check whether the observed allocation is approximately 50/50.

We'll use a chi-square goodness-of-fit test:

H_0:allocation=50/50 

If the SRM test doesn't indicate a problem, we can proceed with the experiment analysis.

So our next actual coding step will be creating the synthetic users and assigning them randomly to A/B.

## Step 5: Generate the user population  -- — User generation & randomization

In [4]:
Before generating events and orders, we first create our 33,600 eligible checkout users.

Our assumptions are:

- Experiment duration: 14 days
- Checkout users/day: 2,400
- Total users: 33,600
- Control: ~16,800
- Treatment: ~16,800
- Randomization: 50/50
- Random seed: 42

SyntaxError: invalid syntax (2825554641.py, line 1)

In [7]:
import pandas as pd
import numpy as np

# Reproducibility
np.random.seed(42)

# Experiment configuration
num_users = 33600

# Create users
users = pd.DataFrame({
    "user_id": [f"U{i:05d}" for i in range(1, num_users + 1)],

    "signup_date": pd.to_datetime(
        np.random.choice(
            pd.date_range("2026-01-01", "2026-08-31"),
            size=num_users
        )
    ),

    "country": np.random.choice(
        ["India", "USA", "UK", "Canada", "Australia"],
        size=num_users,
        p=[0.50, 0.20, 0.12, 0.10, 0.08]
    ),

    "device": np.random.choice(
        ["Mobile", "Desktop", "Tablet"],
        size=num_users,
        p=[0.60, 0.30, 0.10]
    )
})

# Classify users
experiment_start = pd.Timestamp("2026-09-01")

users["user_type"] = np.where(
    users["signup_date"] >= experiment_start - pd.Timedelta(days=30),
    "New",
    "Returning"
)

# Randomize users 50/50
groups = np.array(["A"] * (num_users // 2) +
                   ["B"] * (num_users // 2))

np.random.shuffle(groups)

users["experiment_group"] = groups

users.head()

,user_id,signup_date,country,device,user_type,experiment_group
0,U00001,2026-04-13,USA,Desktop,Returning,A
1,U00002,2026-06-29,Australia,Desktop,Returning,B
2,U00003,2026-04-03,India,Mobile,Returning,A
3,U00004,2026-01-15,Canada,Mobile,Returning,B
4,U00005,2026-04-17,UK,Desktop,Returning,A


In [8]:
users.shape

(33600, 6)

In [9]:
users["country"].value_counts()

country
India        16794
USA           6796
UK            3990
Canada        3341
Australia     2679
Name: count, dtype: int64

In [10]:
users["device"].value_counts()

device
Mobile     20076
Desktop    10285
Tablet      3239
Name: count, dtype: int64

In [11]:
#Now validate the randomization
users["experiment_group"].value_counts()

experiment_group
A    16800
B    16800
Name: count, dtype: int64

In [12]:
users["experiment_group"].value_counts(normalize=True) * 100

experiment_group
A    50.0
B    50.0
Name: proportion, dtype: float64

In [ ]:
# Now let's create the experiment_assignments table.

# We already have users with:

In [13]:
experiments = pd.DataFrame({
    "experiment_id": ["EXP001"],
    "experiment_name": ["New Checkout Experience"],
    "start_date": [pd.Timestamp("2026-09-01")],
    "end_date": [pd.Timestamp("2026-09-14")],
    "hypothesis": [
        "The redesigned checkout will increase checkout conversion"
    ],
    "primary_metric": ["Checkout Conversion Rate"],
    "status": ["Completed"]
})
experiments

,experiment_id,experiment_name,start_date,end_date,hypothesis,primary_metric,status
0,EXP001,New Checkout Experience,2026-09-01,2026-09-14,The redesigned checkout will increase checkout...,Checkout Conversion Rate,Completed


In [14]:
#Take only the information related to the experiment:

In [15]:
experiment_assignments = users[
    ["user_id", "experiment_group"]
].copy()

experiment_assignments["experiment_id"] = "EXP001"

experiment_assignments["assigned_at"] = pd.Timestamp("2026-09-01")
experiment_assignments.head()

,user_id,experiment_group,experiment_id,assigned_at
0,U00001,A,EXP001,2026-09-01
1,U00002,B,EXP001,2026-09-01
2,U00003,A,EXP001,2026-09-01
3,U00004,B,EXP001,2026-09-01
4,U00005,A,EXP001,2026-09-01


In [16]:
#Rearrange the column
experiment_assignments = experiment_assignments[
    [
        "user_id",
        "experiment_id",
        "experiment_group",
        "assigned_at"
    ]
]
experiment_assignments.head()

,user_id,experiment_id,experiment_group,assigned_at
0,U00001,EXP001,A,2026-09-01
1,U00002,EXP001,B,2026-09-01
2,U00003,EXP001,A,2026-09-01
3,U00004,EXP001,B,2026-09-01
4,U00005,EXP001,A,2026-09-01


In [17]:
experiment_assignments["experiment_group"].value_counts()

experiment_group
A    16800
B    16800
Name: count, dtype: int64

In [18]:
experiment_assignments["experiment_group"].value_counts(normalize=True) * 100

experiment_group
A    50.0
B    50.0
Name: proportion, dtype: float64

Now that assignment is ready, we move to the interesting part:

Generate the 14 days of user behavior/events.

We'll create the funnel: Product View → Add to Cart → Checkout → Payment → Purchase

And we'll introduce realistic probabilities so that Treatment B eventually has a higher conversion rate than Control A.

## Step 8: Generate the user behavior/events.

In [19]:
# 2. Generate checkout users first

# Rather than generating every possible event immediately, let's build the funnel sequentially.

import numpy as np
import pandas as pd

np.random.seed(42)

events = []

for _, user in experiment_assignments.iterrows():

    user_id = user["user_id"]
    group = user["experiment_group"]

    # Random date within the 14-day experiment
    event_date = pd.Timestamp("2026-09-01") + pd.Timedelta(
        days=np.random.randint(0, 14)
    )

    # Product View
    if np.random.random() < 0.95:

        events.append([
            user_id,
            "product_view",
            event_date
        ])

        # Add to Cart
        add_cart_prob = 0.50 if group == "A" else 0.51

        if np.random.random() < add_cart_prob:

            events.append([
                user_id,
                "add_to_cart",
                event_date
            ])

            # Checkout Started
            checkout_prob = 0.70 if group == "A" else 0.72

            if np.random.random() < checkout_prob:

                events.append([
                    user_id,
                    "checkout_started",
                    event_date
                ])

In [20]:
events = pd.DataFrame(
    events,
    columns=[
        "user_id",
        "event_name",
        "event_timestamp"
    ]
)

In [21]:
events.head()

,user_id,event_name,event_timestamp
0,U00001,product_view,2026-09-07
1,U00001,add_to_cart,2026-09-07
2,U00002,product_view,2026-09-05
3,U00002,add_to_cart,2026-09-05
4,U00002,checkout_started,2026-09-05


In [22]:
events["event_name"].value_counts()

event_name
product_view        31902
add_to_cart         16155
checkout_started    11441
Name: count, dtype: int64

## Step 9: Generate payment events and orders

We already have users who reached checkout_started. Now we'll simulate what happens after checkout.

1. Payment initiation

Let's take the checkout users:

In [23]:
checkout_users = events[events["event_name"] == "checkout_started"][["user_id", "event_timestamp"]].copy()

In [24]:
# Join their experiment group:
checkout_users = checkout_users.merge(
    experiment_assignments[["user_id", "experiment_group"]],
    on="user_id",
    how="left"
)

2. Generate payment initiation

We'll assume:
- Control → 90% initiate payment
- Treatment → 92% initiate payment

In [25]:
np.random.seed(42)

checkout_users["payment_initiated"] = np.where(
    np.random.random(len(checkout_users)) <
    np.where(
        checkout_users["experiment_group"] == "A",0.90,0.92),1,0)

In [26]:
# Keep only users who initiated payment:
payment_users = checkout_users[
    checkout_users["payment_initiated"] == 1
].copy()

In [27]:
payment_users["payment_success"] = np.where(
    np.random.random(len(payment_users)) <
    np.where(
        payment_users["experiment_group"] == "A",0.95,0.96),1,0)

In [28]:
payment_users["payment_success"].value_counts()

payment_success
1    9940
0     501
Name: count, dtype: int64

In [29]:
# 4. Create the orders table

# Only successful payments can produce completed orders
successful_payments = payment_users[
    payment_users["payment_success"] == 1].copy()

In [30]:
# Generate an order ID:
successful_payments["order_id"] = [
    f"O{i:06d}"
    for i in range(1, len(successful_payments) + 1)
]

In [31]:
# Generate order amount:
successful_payments["order_amount"] = np.round(
    np.random.lognormal(
        mean=3.8,
        sigma=0.6,
        size=len(successful_payments)
    ),
    2
)

In [32]:
#And create a refund flag:
successful_payments["refund_flag"] = (
    np.random.random(len(successful_payments)) < 0.05
).astype(int)

In [33]:
orders = successful_payments[
    [
        "order_id",
        "user_id",
        "event_timestamp",
        "order_amount",
        "refund_flag"
    ]
].copy()

orders.rename(
    columns={"event_timestamp": "order_timestamp"},
    inplace=True
)

## Step 10 — Validate the experiment before analyzing results

In [34]:
# 10.1 Check sample ratio
experiment_assignments["experiment_group"].value_counts()

experiment_group
A    16800
B    16800
Name: count, dtype: int64

In [35]:
# Then calculate percentages:
experiment_assignments["experiment_group"].value_counts(
    normalize=True
) * 100

experiment_group
A    50.0
B    50.0
Name: proportion, dtype: float64

In [36]:
# 10.2 Perform the formal SRM test
from scipy.stats import chisquare

observed = (
    experiment_assignments["experiment_group"]
    .value_counts()
    .sort_index()
    .values
)

total_users = observed.sum()

expected = [
    total_users * 0.50,
    total_users * 0.50
]

chi_stat, p_value = chisquare(
    f_obs=observed,
    f_exp=expected
)

print("Chi-square:", chi_stat)
print("p-value:", p_value)

Chi-square: 0.0
p-value: 1.0


In [37]:
# 10.3 Other data-quality checks

# Before moving to the primary metric, we'll also check:

# Duplicate assignments:- A user should not accidentally have multiple assignments for the same experiment.

In [38]:
experiment_assignments.groupby(
    ["experiment_id", "user_id"]
).size().value_counts()

1    33600
Name: count, dtype: int64

In [39]:
# User appearing in both groups
experiment_assignments.groupby("user_id")["experiment_group"].nunique().value_counts()
#1 → all users this checks the user is in one group, if the user is in both the groups the experiment is contaminated
#Bad practise
# U001 → A
# U001 → B

experiment_group
1    33600
Name: count, dtype: int64

In [40]:
# Next step

# Once SRM passes, we'll calculate our primary metric — Checkout Conversion Rate.

# That's where we'll finally compare:

# Control A vs Treatment B and see whether the redesigned checkout actually improved conversion.

## Step 11 Calculate the Primary Metric

Our primary metric is:

Checkout Conversion Rate (CR)

Definition  CR = Unique users who purchased / Unique users who started checkout
We deliberately use unique users, not number of events.

    For example:

Group	Checkout users	Purchasers	CR

A	5,000	3,000	60%

B	5,000	3,150	63%

CRA = 3000 / 5000  =60%

CRB = 3150 / 5000 = 63%

In [41]:
#First get checkout users:
checkout_users = events[events["event_name"] == "checkout_started"][["user_id","event_timestamp"]].drop_duplicates()
checkout_users.head()

,user_id,event_timestamp
4,U00002,2026-09-05
7,U00003,2026-09-08
19,U00013,2026-09-12
22,U00014,2026-09-02
25,U00015,2026-09-13


In [42]:
#Add their experiment group:

In [43]:
checkout_users = checkout_users.merge(experiment_assignments[["user_id", "experiment_group"]],on="user_id",how="left")
checkout_users.head()

,user_id,event_timestamp,experiment_group
0,U00002,2026-09-05,B
1,U00003,2026-09-08,A
2,U00013,2026-09-12,B
3,U00014,2026-09-02,A
4,U00015,2026-09-13,A


In [44]:
# Now identify purchasers:

purchasers = orders[orders["refund_flag"] == 0][["user_id"]].drop_duplicates()

In [45]:
# Mark whether each checkout user purchased:
checkout_users["purchased"] = (checkout_users["user_id"].isin(purchasers["user_id"]).astype(int))

In [46]:
conversion_summary = ( checkout_users.groupby("experiment_group").agg(checkout_users=("user_id", "nunique"),purchasers=("purchased", "sum")))

conversion_summary["conversion_rate"] = ( conversion_summary["purchasers"]/ conversion_summary["checkout_users"])

conversion_summary

,checkout_users,purchasers,conversion_rate
experiment_group,,,
A,5570,4502,0.808259
B,5871,4928,0.839380


For our project's primary metric, we should decide whether a refunded order counts as a "purchase."

For a typical checkout conversion experiment, I'd recommend:

A successful completed order counts as a conversion, even if it is later refunded.

Refunds should be analyzed separately as a guardrail metric.

So when we calculate CR, we should actually use all completed orders, not remove refunded orders.

We'll fix that when we run the final metric calculation.

Now we have the observed CR for A and B. The next question is the important one:
Is the difference between A and B statistically significant, or could it simply be random variation?
That's where we'll perform the two-proportion z-test, calculate the p-value and 95% confidence interval.

In [47]:
control_cr = conversion_summary.loc["A", "conversion_rate"]
treatment_cr = conversion_summary.loc["B", "conversion_rate"]

absolute_lift = treatment_cr - control_cr

relative_lift = (
    (treatment_cr - control_cr)
    / control_cr
)

print("Control CR:", control_cr)
print("Treatment CR:", treatment_cr)
print("Absolute lift:", absolute_lift)
print("Relative lift:", relative_lift)

Control CR: 0.8082585278276481
Treatment CR: 0.8393800034065747
Absolute lift: 0.03112147557892664
Relative lift: 0.038504357835322384


1.Treatment B has a higher observed conversion rate.
2.Absolute lift = 3.11 percentage points

83.94%-80.83%=3.11% 

We normally describe this as:

Treatment increased conversion by 3.11 percentage points.

3. Relative lift = 3.85%

(83.94−80.83) ×100  / 80.83 =3.85

4.But here's the important part

We cannot say Treatment B is statistically better yet.

We only know that: Observed difference = +3.11 percentage points

We now need to determine whether this difference could reasonably have occurred due to random chance.

That's exactly what the two-proportion z-test will answer.

## Novely and primacy effect
For our 14-day experiment, we can simulate this.

Our planned behavior

Instead of making Treatment B perform at its "normal" level from Day 1:

Day	Treatment behavior
Day 1	Affected by novelty/primacy
Day 2	Affected
Day 3	Affected
Day 4 onward	Stable / normal behavior

We could simulate:

Day 1 → 66%
Day 2 → 65%
Day 3 → 64%
Day 4 → 63%
Day 5 → 63%
...
Day 14 → 63%

We should monitor daily conversion throughout the experiment, even though our primary analysis is Days 4–14.

In real-time A/B testing, we usually do NOT automatically remove the data from the novelty/primacy period.

The first 3 days may show a novelty effect. But those users are real users who genuinely experienced the treatment, so their data is still valid data.

We don't simply say: "These results aren't what we expected, so let's delete them."

There are two legitimate approaches, depending on the experiment design.

Approach 1 — Include all data, If the question is: "What is the overall impact of launching this feature?"

Then we should include the entire experiment period.

The novelty effect is part of the real-world impact.

Approach 2 — Predefine a ramp-up/stabilization period

If we have a strong reason to believe users need time to adapt to the new experience, we can define before the experiment starts:

"The first 3 days are considered a predefined stabilization period; the primary analysis will evaluate Days 4–14."

The critical point is: We decide this before looking at the results.

## Step 13 — Analyze the experiment using the stable period.

We decided:
- Experiment duration: 14 days
- Days 1–3: predefined stabilization period
- Days 4–14: primary analysis window
- We still retain and monitor Days 1–3.

In [48]:
# First, we need to make sure our data actually has dates
# Our current checkout_users has event_timestamp, so let's create a date column

In [49]:
checkout_users["date"] = pd.to_datetime(checkout_users["event_timestamp"]).dt.date

In [50]:
checkout_users.head()

,user_id,event_timestamp,experiment_group,purchased,date
0,U00002,2026-09-05,B,1,2026-09-05
1,U00003,2026-09-08,A,0,2026-09-08
2,U00013,2026-09-12,B,1,2026-09-12
3,U00014,2026-09-02,A,0,2026-09-02
4,U00015,2026-09-13,A,1,2026-09-13


In [51]:
# Now check the daily volume:
daily_users = ( checkout_users.groupby(["date", "experiment_group"])["user_id"].nunique().reset_index())

daily_users.head()

,date,experiment_group,user_id
0,2026-09-01,A,390
1,2026-09-01,B,389
2,2026-09-02,A,430
3,2026-09-02,B,418
4,2026-09-03,A,392


In [52]:
# Then calculate daily conversion, We can calculate daily purchasers:
daily_conversion = ( checkout_users .groupby(["date", "experiment_group"]).agg(checkout_users=("user_id", "nunique"),purchasers=("purchased", "sum")).reset_index())

daily_conversion["conversion_rate"] = (daily_conversion["purchasers"]/ daily_conversion["checkout_users"])
daily_conversion.head()

,date,experiment_group,checkout_users,purchasers,conversion_rate
0,2026-09-01,A,390,322,0.825641
1,2026-09-01,B,389,341,0.876607
2,2026-09-02,A,430,356,0.827907
3,2026-09-02,B,418,344,0.822967
4,2026-09-03,A,392,325,0.829082


In [53]:
# Then we'll filter the primary analysis window
checkout_users["date"] = pd.to_datetime(checkout_users["event_timestamp"]).dt.normalize()
analysis_data = checkout_users[checkout_users["date"] >= pd.Timestamp("2026-09-04")].copy()

But don't run the z-test yet.

First, let's inspect the daily conversion numbers and make sure our synthetic data actually shows the pattern we're trying to model.

Next step after this: calculate the final Control vs Treatment conversion rate for Days 4–14, then perform the two-proportion z-test.

In [54]:
# Control vs Treatment conversion
analysis_summary = ( analysis_data.groupby("experiment_group").agg(checkout_users=("user_id", "nunique"),purchasers=("purchased", "sum")))

analysis_summary["conversion_rate"] = (analysis_summary["purchasers"] / analysis_summary["checkout_users"])

analysis_summary

,checkout_users,purchasers,conversion_rate
experiment_group,,,
A,4358,3499,0.802891
B,4645,3897,0.838967


In [55]:
# calculate the lift:
control_cr = analysis_summary.loc["A", "conversion_rate"]
treatment_cr = analysis_summary.loc["B", "conversion_rate"]

absolute_lift = treatment_cr - control_cr
relative_lift = absolute_lift / control_cr

print("Control CR:", control_cr)
print("Treatment CR:", treatment_cr)
print("Absolute Lift:", absolute_lift)
print("Relative Lift:", relative_lift)

Control CR: 0.8028912345112437
Treatment CR: 0.8389666307857911
Absolute Lift: 0.03607539627454748
Relative Lift: 0.044931859664040556


## Next Step 12 - Statistical significance

In [56]:
from statsmodels.stats.proportion import proportions_ztest

count = analysis_summary["purchasers"].values
nobs = analysis_summary["checkout_users"].values

z_stat, p_value = proportions_ztest(count,nobs)

print("Z-statistic:", z_stat)
print("P-value:", p_value)

Z-statistic: -4.467194047554989
P-value: 7.925223717010684e-06


In [78]:
# 0.000007925 < 0.05
#The treatment has a statistically significant difference in checkout conversion compared with control.

In [58]:
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.proportion import confint_proportions_2indep

control_purchases = analysis_summary.loc["A", "purchasers"]
control_users = analysis_summary.loc["A", "checkout_users"]

treatment_purchases = analysis_summary.loc["B", "purchasers"]
treatment_users = analysis_summary.loc["B", "checkout_users"]

ci_low, ci_high = confint_proportions_2indep( treatment_purchases, treatment_users,control_purchases,control_users,method="wald")

print("95% CI:", ci_low, ci_high)

95% CI: 0.020225175999653913 0.05192561654944104


That means: The true treatment improvement is estimated to be between 0.2 and 0.05 percentage points.

Because the entire CI is above 0, the treatment is statistically significant at the 5% level.

After this, we'll do the final important step:

Statistical significance vs. practical significance (our 5% MDE).

Statistical significance vs. practical significance

In [59]:
# We defined our MDE = 5% relative improvement

In [60]:
mde = 0.05  # 5% relative improvement

observed_relative_lift = (treatment_cr - control_cr) / control_cr

print("Observed relative lift:", observed_relative_lift)
print("MDE:", mde)

if observed_relative_lift >= mde:
    print("Treatment meets the practical significance threshold.")
else:
    print("Treatment does NOT meet the practical significance threshold.")

Observed relative lift: 0.044931859664040556
MDE: 0.05
Treatment does NOT meet the practical significance threshold.


The treatment improved conversion, but did not reach the minimum business improvement we decided was worthwhile. So even if the Z-test shows statistical significance, we would not automatically recommend rollout based on the primary metric alone.

## Next step  13 → Guardrail metrics

Now we check whether the treatment caused any negative side effects:

- Payment failure rate
- Checkout abandonment rate
- Refund rate

This is important because a higher conversion rate isn't useful if, for example, payment failures or refunds also increase.

In [61]:
payment_failure = ( payment_users.groupby("experiment_group").agg(payment_attempts=("user_id", "nunique"),
        failed_payments=("payment_success", lambda x: (x == 0).sum()) ))

payment_failure["failure_rate"] = (payment_failure["failed_payments"]/ payment_failure["payment_attempts"])

payment_failure

,payment_attempts,failed_payments,failure_rate
experiment_group,,,
A,5019,259,0.051604
B,5422,242,0.044633


Here, treatment would actually be better, because fewer payments failed.
For a guardrail, we're mainly checking:

Did the new checkout experience make payment quality worse?
After this, we'll calculate checkout abandonment rate.

Checkout Abandonment Rate.

The formula is:Abandonment Rate= (Checkout Users−Purchasers) / Checkout Users
- Since checkout_users already contains the purchased flag:

In [62]:
abandonment = ( analysis_data.groupby("experiment_group").agg(checkout_users=("user_id", "nunique"),purchasers=("purchased", "sum")))

abandonment["abandoned_checkouts"] = (abandonment["checkout_users"]- abandonment["purchasers"])

abandonment["abandonment_rate"] = (abandonment["abandoned_checkouts"]/ abandonment["checkout_users"])

abandonment

,checkout_users,purchasers,abandoned_checkouts,abandonment_rate
experiment_group,,,,
A,4358,3499,859,0.197109
B,4645,3897,748,0.161033


That would mean the new checkout reduced abandonment by 3 percentage points.

So far we're checking:
- ✅ Primary metric: Conversion
- ✅ Statistical significance
- ✅ Practical significance
- ✅ Payment failure rate
- 🔄 Checkout abandonment rate — now
- ⏭️ Refund rate
- ⏭️ Final experiment conclusion / recommendation

## Refund Rate guardrail

In [63]:
# We want to check whether the new checkout experience increased the proportion of completed orders that were later refunded.

In [64]:
refund_summary = (orders.groupby("user_id").agg(orders=("order_id", "nunique"),refunded_orders=("refund_flag", "sum")))

refund_summary["refund_rate"] = (refund_summary["refunded_orders"]/ refund_summary["orders"])

In [65]:
# But we need the experiment group, so merge it:
refund_summary = refund_summary.reset_index().merge(experiment_assignments[["user_id", "experiment_group"]],on="user_id",how="left")

In [66]:
refund_rate = (refund_summary.groupby("experiment_group").agg(total_orders=("orders", "sum"),refunded_orders=("refunded_orders", "sum")) )

refund_rate["refund_rate"] = (refund_rate["refunded_orders"]/ refund_rate["total_orders"])

refund_rate

,total_orders,refunded_orders,refund_rate
experiment_group,,,
A,4760,258,0.054202
B,5180,252,0.048649


In [67]:
# We want: Treatment refund rate ≤ Control refund rate

# If treatment has higher conversion but also substantially higher refunds, that could be a warning sign.

# After this, we'll bring all metrics together into one final experiment-results table and make the rollout recommendation.

In [68]:
# However, remember our primary metric is the deciding metric:

# Statistical significance: ✅
# Relative lift: 4.49%
# MDE: 5%
# Practical significance: ❌

# So despite the guardrails looking healthy, the treatment didn't meet our predefined business threshold.

# Next step

# Let's calculate the secondary metric: Revenue per exposed user (RPEU).

# This will tell us whether the treatment generates more revenue per checkout user, even though its conversion lift didn't reach the MDE.

In [69]:
# Next, let's calculate the secondary metric: Revenue Per Exposed User (RPEU).

# For our experiment, an exposed user is a user who started checkout.

# RPEU = Total Revenue / Unique Checkout Users

# Use the Day 4–14 analysis period so all our primary/secondary analysis uses the same window.

In [70]:
# Get users who started checkout during Day 4–14
analysis_users = analysis_data[ ["user_id", "experiment_group"]].drop_duplicates()

# Get their orders
analysis_revenue = analysis_users.merge( orders[["user_id", "order_amount"]],on="user_id",how="left")

analysis_revenue["order_amount"] = ( analysis_revenue["order_amount"].fillna(0))

revenue_summary = ( analysis_revenue.groupby("experiment_group").agg(checkout_users=("user_id", "nunique"),total_revenue=("order_amount", "sum")))

revenue_summary["revenue_per_user"] = ( revenue_summary["total_revenue"] / revenue_summary["checkout_users"])

revenue_summary

,checkout_users,total_revenue,revenue_per_user
experiment_group,,,
A,4358,197962.82,45.425154
B,4645,217546.05,46.834456


In [71]:
control_rpeu = revenue_summary.loc["A", "revenue_per_user"]
treatment_rpeu = revenue_summary.loc["B", "revenue_per_user"]

rpeu_lift = ((treatment_rpeu - control_rpeu) / control_rpeu)

print("Control RPEU:", control_rpeu)
print("Treatment RPEU:", treatment_rpeu)
print("RPEU Relative Lift:", rpeu_lift)

Control RPEU: 45.42515374024782
Treatment RPEU: 46.83445640473627
RPEU Relative Lift: 0.031024719752126517


In [72]:
# let's filter orders to the same Day 4–14 analysis window before interpreting RPEU
analysis_orders = orders[
    pd.to_datetime(orders["order_timestamp"]) >= pd.Timestamp("2026-09-04")
].copy()

analysis_orders = analysis_orders[
    pd.to_datetime(analysis_orders["order_timestamp"]) <= pd.Timestamp("2026-09-14")
].copy()

In [73]:
# calculate revenue per checkout user:
analysis_revenue = analysis_data[
    ["user_id", "experiment_group"]
].drop_duplicates().merge(
    analysis_orders[["user_id", "order_amount"]],
    on="user_id",
    how="left"
)

analysis_revenue["order_amount"] = (
    analysis_revenue["order_amount"].fillna(0)
)

revenue_summary = (
    analysis_revenue
    .groupby("experiment_group")
    .agg(
        checkout_users=("user_id", "nunique"),
        total_revenue=("order_amount", "sum")
    )
)

revenue_summary["revenue_per_user"] = (
    revenue_summary["total_revenue"]
    / revenue_summary["checkout_users"]
)

revenue_summary

,checkout_users,total_revenue,revenue_per_user
experiment_group,,,
A,4358,197962.82,45.425154
B,4645,217546.05,46.834456


In [74]:
control_rpeu = revenue_summary.loc["A", "revenue_per_user"]
treatment_rpeu = revenue_summary.loc["B", "revenue_per_user"]

rpeu_lift = (
    (treatment_rpeu - control_rpeu)
    / control_rpeu
)

print("Control RPEU:", control_rpeu)
print("Treatment RPEU:", treatment_rpeu)
print("RPEU Relative Lift:", rpeu_lift)

Control RPEU: 45.42515374024782
Treatment RPEU: 46.83445640473627
RPEU Relative Lift: 0.031024719752126517


In [75]:
# It's a positive secondary-metric result, but we haven't tested whether the RPEU difference is statistically significant yet.
# For our project, we'll use Mann–Whitney U test for the revenue distribution because revenue is typically skewed.
# Next step → test whether the RPEU/revenue difference is statistically significant.

Why Mann–Whitney U for revenue?

Most customers spend relatively small amounts, while a few customers spend a lot. This creates a right-skewed distribution.

A traditional independent t-test relies more heavily on assumptions about the distribution and mean. Mann–Whitney U is a non-parametric test, so it doesn't require the revenue values themselves to be normally distributed.

It tests whether the distributions of the two groups differ.

In [76]:
from scipy.stats import mannwhitneyu

control_revenue = analysis_revenue[analysis_revenue["experiment_group"] == "A"]["order_amount"]

treatment_revenue = analysis_revenue[analysis_revenue["experiment_group"] == "B"]["order_amount"]

u_stat, p_value = mannwhitneyu( control_revenue,treatment_revenue,alternative="two-sided")

print("U-statistic:", u_stat)
print("P-value:", p_value)

U-statistic: 9867173.0
P-value: 0.03885911178165815


0.03886 < 0.05

Interpretation , We reject the null hypothesis.

So there is a statistically significant difference in the revenue distributions between Control and Treatment for the Sep 4–14 analysis window.

But there's an important distinction

We didn't define an MDE for RPEU in our project. Our predefined 5% MDE was for the primary conversion metric.

Therefore, we should not say RPEU failed the 5% MDE. We simply report:

"Treatment showed a statistically significant positive difference in revenue per exposed user, with a 3.1% relative increase."

Why?

The primary KPI is the most important metric, and although the treatment produced a statistically significant +4.49% relative conversion lift, it did not reach our predefined 5% MDE.

So the conclusion would be:

Treatment shows promising results, but the improvement is slightly below the predefined business threshold. Keep the experiment/consider further testing rather than immediately rolling it out.

This is actually a very good A/B testing project outcome because you're demonstrating that statistical significance ≠ practical significance.

Next Step 13 :

In [79]:
analysis_revenue.columns

Index(['user_id', 'experiment_group', 'order_amount'], dtype='str')

In [80]:
checkout_users.columns

Index(['user_id', 'event_timestamp', 'experiment_group', 'purchased', 'date'], dtype='str')

In [81]:
# And we'll use MySQL for:

# Data storage
# SQL transformations
# KPI calculations
# Daily experiment metrics
# Control vs Treatment comparisons
# Preparing tables/views for Power BI

In [85]:
checkout_users.to_csv("checkout_users.csv", index=False)

In [87]:
analysis_revenue.to_csv("analysis_revenue.csv",index=False)

In [93]:
payment_users.to_csv('payment_users.csv',index=False)

In [95]:
orders.to_csv("orders.csv",index=False)

Index(['user_id', 'experiment_group', 'order_amount'], dtype='str')

In [99]:
analysis_data.to_csv("analysis_data.csv",index=False)